* NeuroVFM


#### Goals
- encoder pretraining이 Vol-JEPA라는 점 확인
- frozen encoder 위에서 study-level MIL head가 학습되는지 확인
- `embed`와 `predict`가 토큰 embedding과 multi-label 확률을 내는지 확인


#### Contents
1. Stack
    <br>1.1 Vol-JEPA pretraining
    <br>1.2 Classify-Then-Aggregate head
2. Experiment
    <br>2.1 합성 볼륨으로 두 단계 학습

NeuroVFM(Kondepudi et al., Nature Medicine 2026)은 세 층이다.

1. unlabeled CT/MRI에 Vol-JEPA로 3D ViT를 사전학습
2. encoder를 얼리고 Classify-Then-Aggregate MIL로 study-level 진단 (공개 모델은 MRI 74 / CT 82라벨)
3. frozen encoder 위에 findings LLM(Qwen3-8B)을 SFT

3번은 언어모델 미세조정이라 여기서 다시 만들지 않는다. 가중치는 [mlinslab/neurovfm-encoder](https://huggingface.co/mlinslab/neurovfm-encoder) 승인이 필요하고, 이 노트북의 파라미터는 랜덤 초기화다. 연구용 구조이며 임상 판단에 쓰지 않는다.


## 1. Stack


In [1]:
import sys
from pathlib import Path

import torch

candidates = [Path.cwd(), Path.cwd() / "jepa"]
for cand in candidates:
    module_dir = cand / "modules"
    if module_dir.is_dir():
        sys.path.insert(0, str(module_dir))
        break
else:
    raise FileNotFoundError("jepa/modules 를 찾지 못했다. 노트북 cwd 를 ai-lab 또는 ai-lab/jepa 로 둔다.")

torch.manual_seed(0)
device = torch.device("cpu")
print("device", device)

from neurovfm import NeuroVFM

num_labels = 4
model = NeuroVFM(num_labels=num_labels, patch_size=(2, 4, 4), depth=2, num_heads=4).to(device)
volume = torch.rand(2, 1, 8, 32, 32, device=device)
labels = torch.randint(0, 2, (volume.size(0), num_labels), device=device).float()
embeddings = model.embed(volume)
print("embeddings", tuple(embeddings.shape))
print("probabilities", tuple(model.predict(volume).shape))


device cpu
embeddings (2, 256, 96)
probabilities (2, 4)


#### 1.1 Vol-JEPA pretraining

encoder와 predictor만 업데이트한다. diagnostic head는 이 단계의 loss에 들어가지 않는다.


In [2]:
pretrain_opt = torch.optim.AdamW(model.pretraining.trainable_parameters(), lr=1e-3, weight_decay=0.05)
pretrain_loss = []
for step in range(3):
    pretrain_opt.zero_grad()
    loss = model.pretrain_loss(volume)
    loss.backward()
    pretrain_opt.step()
    model.update_target(momentum=0.994)
    pretrain_loss.append(loss.item())
print("vol-jepa", [round(v, 4) for v in pretrain_loss])


vol-jepa [0.3906, 0.3089, 0.2677]


#### 1.2 Classify-Then-Aggregate head

토큰마다 라벨 logit을 만들고, 라벨별 attention으로 study logit을 합친다. encoder embedding은 detach해서 head만 학습한다. 공식 loss는 BCE다.


In [3]:
model.freeze_encoder()
for param in model.encoder.parameters():
    param.grad = None
head_opt = torch.optim.AdamW(model.diagnostic_head.parameters(), lr=1e-3, weight_decay=0.05)
cls_loss = []
for step in range(6):
    head_opt.zero_grad()
    loss = model.classification_loss(volume, labels)
    loss.backward()
    head_opt.step()
    cls_loss.append(loss.item())

encoder_grad = any(p.grad is not None for p in model.encoder.parameters())
print("bce", [round(v, 4) for v in cls_loss])
print("encoder received grad", encoder_grad)
embeddings = model.embed(volume)
probs = model.predict(volume)
print("embedding mean abs diff", round((embeddings[0] - embeddings[1]).abs().mean().item(), 4))
print("prob max abs diff", round((probs[0] - probs[1]).abs().max().item(), 4))
print(probs.detach().cpu())


bce [0.6925, 0.6803, 0.6681, 0.6554, 0.642, 0.6277]
encoder received grad False
embedding mean abs diff 0.0507
prob max abs diff 0.0001
tensor([[0.5844, 0.5906, 0.5038, 0.5007],
        [0.5844, 0.5906, 0.5038, 0.5007]])


## 2. Experiment

#### 2.1 읽는 법

`probs`는 학습된 진단이 아니다. 라벨도 볼륨에서 만든 것이 아니라 무작위 multi-hot이다. 확인하는 것은 단계 순서다. Vol-JEPA로 token embedding을 만들고, 그 embedding을 멈춘 뒤 MIL head가 study 확률을 낸다.

실제 CT study를 넣으려면 공식 `StudyPreprocessor.load_study`와 승인된 encoder 가중치가 필요하다. ich-vlm의 `notebook/neurovfm_tutorial.ipynb`가 그 경로다.
